# 05 — Ensemble learning

**Estimated time:** 110–150 minutes  
**Prerequisites:** notebooks 00–04; bagging, boosting, and CV.  
**Depends on:** leakage-safe preprocessing and development-only model selection.

## Learning objectives

- Position Random Forest as a bagging baseline against boosting.
- Measure model diversity through out-of-fold error correlation.
- Build soft voting and leakage-safe stacking.
- Compare performance, calibration, latency, and operational complexity.


In [ ]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# Known interoperability/UI warnings do not affect predictions or notebook execution.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (balanced_accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, log_loss, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # Return the course root when present, otherwise the notebook's directory.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # Set FAST_MODE=0 for full-size experiments; laptop mode is the default.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # The course ships with a local dataset; notebooks never access the network.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # Load the data, encode y, and exclude post-call duration by default.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # Deterministic stratified 60/20/20 split; test stays sealed until notebook 09.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # Preprocessing is fitted only inside the enclosing training/CV pipeline.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"log_loss": log_loss(y_true, probability),
            "brier_score": brier_score_loss(y_true, probability),
            "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
BRAND_COLOR = "#2a9d8f"
ACCENT_COLOR = "#e76f51"
sns.set_theme(style="whitegrid", context="notebook", palette=[BRAND_COLOR, ACCENT_COLOR, "#264653", "#f4a261"])
plt.rcParams.update({
    "axes.facecolor": "#fcfcfc",
    "figure.facecolor": "white",
    "axes.edgecolor": "#d9d9d9",
    "grid.color": "#e8e8e8",
    "grid.linewidth": 0.8,
    "axes.titleweight": "bold",
    "axes.labelweight": "medium",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})


In [ ]:
import time
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

development, validation, _sealed_test = make_splits(load_bank_data(), reduced=FAST_MODE)
X_dev, y_dev = split_xy(development)
X_val, y_val = split_xy(validation)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")

def pipe(model, scale_numeric=False):
    return Pipeline([("preprocess", make_preprocessor(development, scale_numeric=scale_numeric)),
                     ("model", model)])

logistic = pipe(LogisticRegression(max_iter=1200, random_state=SEED), scale_numeric=True)
forest = pipe(RandomForestClassifier(
    n_estimators=180 if FAST_MODE else 500, min_samples_leaf=4,
    max_features="sqrt", class_weight="balanced_subsample",
    n_jobs=-1, random_state=SEED,
))
boosting = pipe(lgb.LGBMClassifier(
    n_estimators=240 if FAST_MODE else 500, learning_rate=.04,
    num_leaves=31, min_child_samples=40, subsample=.9,
    colsample_bytree=.9, random_state=SEED, n_jobs=-1, verbosity=-1,
))


Random Forest averages decorrelated deep trees trained on bootstrap samples: variance falls, but
probability estimates can be coarse and inference cost grows with tree count. Boosting fits trees
sequentially to correct current errors: it often improves tabular accuracy but can overfit without
shrinkage and structural regularization.

Ensembling helps when models are individually competent and make different errors. Measuring
diversity on in-sample predictions is optimistic, so we generate out-of-fold (OOF) probabilities.


In [ ]:
base = {"logistic": logistic, "random_forest": forest, "lightgbm": boosting}
oof = {}
for name, estimator in base.items():
    oof[name] = cross_val_predict(estimator, X_dev, y_dev, cv=cv,
                                  method="predict_proba", n_jobs=-1)[:, 1]
oof_frame = pd.DataFrame(oof)
display(oof_frame.corr())
error_frame = oof_frame.sub(y_dev.to_numpy(), axis=0)
display(error_frame.corr().style.format("{:.3f}"))


## Soft voting and stacking

Soft voting averages probabilities and assumes they are on compatible scales; uncalibrated members
can dominate. Stacking trains a meta-model on OOF base predictions. `StackingClassifier(cv=...)`
generates those OOF predictions internally, then refits base learners on all development rows.
Passing predictions fitted on the same rows would leak base-model overfit into the meta-model.


In [ ]:
voting = VotingClassifier(
    estimators=[("lr", logistic), ("rf", forest), ("lgb", boosting)],
    voting="soft", n_jobs=-1,
)
stacking = StackingClassifier(
    estimators=[("lr", logistic), ("rf", forest), ("lgb", boosting)],
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    cv=cv, stack_method="predict_proba", passthrough=False, n_jobs=-1,
)
candidates = {**base, "soft_vote": voting, "stack": stacking}


In [ ]:
rows, fitted = [], {}
for name, estimator in candidates.items():
    start = time.perf_counter()
    fitted[name] = estimator.fit(X_dev, y_dev)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    p = fitted[name].predict_proba(X_val)[:, 1]
    infer_ms_per_1k = (time.perf_counter() - start) * 1000 / len(X_val) * 1000
    rows.append({"model": name, **classification_metrics(y_val, p),
                 "fit_seconds": fit_seconds, "inference_ms_per_1k": infer_ms_per_1k})
ensemble_results = pd.DataFrame(rows).set_index("model")
ensemble_results.sort_values("cost", ascending=True)


In [ ]:
from sklearn.calibration import CalibrationDisplay
fig, ax = plt.subplots(figsize=(6, 5), dpi=140)
calibration_colors = [BRAND_COLOR, ACCENT_COLOR, "#264653", "#f4a261"]
for color, name in zip(calibration_colors, ["random_forest", "lightgbm", "soft_vote", "stack"]):
    display_curve = CalibrationDisplay.from_predictions(
        y_val, fitted[name].predict_proba(X_val)[:, 1], n_bins=8,
        strategy="quantile", name=name, ax=ax,
    )
    display_curve.line_.set_color(color)
    display_curve.line_.set_linewidth(2.2)
ax.set_title("Validation calibration", pad=14, fontsize=14, fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()


Stacking is not free: it multiplies training and inference paths, complicates monitoring and
explanations, and can fail when base predictions shift differently. Prefer the simplest model
inside an uncertainty-aware performance tolerance. If stacking's gain is tiny, operationally it
is usually a loss.

## Common mistakes and leakage warnings

- Training the meta-model on in-sample base predictions.
- Comparing an ensemble to weak or under-tuned individual baselines.
- Averaging badly calibrated probabilities without inspection.
- Ignoring correlated failures, latency, artifact size, and dependency count.
- Selecting the ensemble after repeated test comparisons.

## Exercises

1. Optimize voting weights using development OOF predictions only.
2. Calibrate each base learner, then reassess voting.
3. **Challenge:** create a nested-CV stacking evaluation and compare its confidence interval with
   LightGBM's. Recommend a production model using an explicit complexity penalty.

## Summary

Random Forest supplies a robust bagging reference; boosting often has stronger ranking. OOF
predictions make diversity analysis and stacking honest. Ensembles earn deployment only when a
reliable gain outweighs additional latency, explanation, monitoring, and maintenance cost.

## References

- [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html)
- [StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)
